# Generating Feature and Label Tiles from Classified LiDAR and Vector Data

This notebook demonstrates how to process a large collection of classified LiDAR files (.las / .laz) into individual feature tiles derived from LiDAR data, and how to generate corresponding raster label tiles from vector data.  
Additional preprocessing steps such as High Pass Median Filtering (HPMF) and rasterization with vector buffering are applied to enhance terrain features and create realistic training datasets for machine learning.

## Workflow:

---

1. **File search**  
   Recursively search for all `.las` and `.laz` files in the working directory or sub-folders. If the data are in zip file, they are first extracted
   
---

2. **DEM parameters**  
   Define interpolation settings such as resolution, output type (IDW), search radius, power parameter, and window size.
   
---

3. **Per-file processing**  
   For each LiDAR file:
   - Read only ground-classified points (classification code = 2).  
   - Apply statistical outlier removal to clean the point cloud.  
   - Interpolate to raster with Inverse Distance Weighting (IDW), creating one Digital Elevation Model (DEM) tile per input file.
     
---

4. **High Pass Median Filter (HPMF)**  
   Apply a high-pass median filter to the DEM tiles to highlight local elevation changes.  
   The method subtracts each cell value from the median of its neighbourhood, emphasizing fine-scale terrain variability while suppressing broad trends.
   
---

5. **Global robust statistics computation**   
    To ensure consistent and robust scaling across all tiles:
   - Each raster tile is read block by block to avoid memory overload.
   - Invalid values (e.g., NaN, -9999) are masked out.
   - Up to a fixed number (e.g., 5000) of valid pixels are randomly sampled per tile.
   - The sampled values from all tiles are aggregated into a global pool.
   - From this global pool, robust statistics are computed: Lower percentile (p1), Upper percentile (p99), Median.
   These global values are then used for consistent scaling or normalization of all tiles, ensuring that extreme outliers in individual tiles do not distort the global distribution.

---

6. **Rasterization**  
   Each DEM tile is used as a mask to extract the matching area from a large vector dataset and rasterize it into a label tile.  
   Before rasterization, vector geometries are buffered (1.5 m) to ensure coverage of narrow or thin features.  
   The buffered geometries are then rasterized onto the DEM grid.  

   To avoid creating unrealistic fixed-width labels, the rasterized geometries are combined with the HPMF output:  
   only pixels within the buffered vector lines **and** with an HPMF value below –0.075 are kept as ditch pixels.
   This ensures that labels follow real terrain depressions rather than forming uniform strips around the vector lines **(we can try different thresholds)**.

   Finally, a majority filter is applied to remove isolated spurious pixels and smooth the label shapes.  
   
---

7. **Output**  
   Results are written to dedicated subdirectories:  
   - `model_input_data/dem_tiles` -> DEM rasters generated from IDW interpolation  
   - `model_input_data/hpmf_tiles` -> DEM rasters after hierarchical progressive morphological filtering  
   - `model_input_data/rasters` -> Final standardized rasterized tiles prepared for machine learning

    
    These standardized tiles can later be mosaicked or subdivided further into training patches for the ML workflow.

---

The tile-based approach is well-suited for handling very large LiDAR datasets, since each file is processed independently without overloading RAM.


## Environment Setup and Imports

On Windows I recommend to use a dedicated **Conda environment** for this workflow, because installing **PDAL** and its dependencies can be problematic on Windows.
With Conda, installation is much simpler since most geospatial libraries (PDAL, GDAL, etc.) are available via the `conda-forge` channel.

Example environment creation:

```bash
conda create -n lidar-env python=3.11 -c conda-forge pdal numpy jupyter pathlib json
conda activate lidar-env
```

Once the environment is active, you can import the necessary Python libraries in the notebook:

In [10]:
import os
import zipfile
import geopandas as gpd
from shapely.geometry import box
import rasterio
from rasterio import features
import numpy as np
import math
import csv
from whitebox.whitebox_tools import WhiteboxTools
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import median_filter, generic_filter
from scipy import stats
import cv2
wbt = WhiteboxTools()
wbt.verbose = False

### LiDAR Preprocessing and DEM Generation

File search

In [11]:
# Current working directory absolute path
curr_dir = Path().resolve()
# Data directory as a sub-folder (where I have the data, change if needed)
lidar_data_dir = curr_dir / "data" / "lidar_data"
# Recursively find all ZIP files
zip_files = list(lidar_data_dir.rglob("*.zip"))
print(f"Found {len(zip_files)} ZIP files.")
for zip_path in zip_files:
    # Create a subfolder with the same name as the ZIP file (without extension)
    extract_dir = zip_path.parent / zip_path.stem
        # Skip if already extracted
    if extract_dir.exists():
        print(f"⏩ Skipping {zip_path.name} (already extracted)")
        continue
    # Extract all contents of the ZIP file
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"Extracted: {zip_path.name} → {extract_dir}")
    
# Recursive search of *.laz or *.las files in all sub-folders
las_files = list(lidar_data_dir.rglob("*.laz")) + list(lidar_data_dir.rglob("*.las"))
las_files = [str(f) for f in las_files]
print(f"Found total of {len(las_files)} .las/.laz files")

Found 8 ZIP files.
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T101359216523441.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T102630519832085.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250904T104027841004545.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T075609555918793.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T081302777824096.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T082522557653894.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T092819382121549.zip (already extracted)
⏩ Skipping Maanmittauslaitos_Laserkeilausaineisto_5p_20250908T094524692517539.zip (already extracted)
Found total of 1600 .las/.laz files


Define DEM parameters

In [12]:
# DEM parameters
resolution = 0.5
output_type = "min" # or "idw", but "min" probably better for our purpose
radius = 1.0 # 1 m, we can try different values
power = 2.0 # only for idw
window_size = 5 # if there are no points in radius, how many surrounding values use for interpolation

Set output directiories

In [13]:
data_dir = lidar_data_dir.parent
dem_dir = data_dir / "dem_data"
hpmf_dir = data_dir /  "hpmf_tiles"
label_dir = data_dir / "label_tiles"
normalized_dir = data_dir / "normalized_tiles"

# Create data folders if they don't exist
dem_dir.mkdir(parents=True, exist_ok=True)
hpmf_dir.mkdir(parents=True, exist_ok=True)
label_dir.mkdir(parents=True, exist_ok=True)
normalized_dir.mkdir(parents=True, exist_ok=True)

Parallel processing of .laz files using PDAL pipeline and saving as DEM .tif files

In [ ]:
def run_pdal(las):
    las_path = Path(las)
    dem_file = dem_dir / f"{las_path.stem}_dem_{output_type}.tif"
    if dem_file.exists():
        return f"⏩ {las_path.name} skipped"

    # Create pipeline JSON dynamically
    pipeline_dict = {
        "pipeline": [
            {"type": "readers.las", "filename": str(las_path)},
            {"type": "filters.range", "limits": "Classification[2:2]"},
            {"type": "filters.outlier", "method": "statistical", "mean_k": 8, "multiplier": 2.5},
            {
                "type": "writers.gdal",
                "filename": str(dem_file),
                "resolution": resolution,
                "output_type": output_type,
                "radius": radius,
                "power": power,
                "window_size": window_size,
                "gdaldriver": "GTiff"
            }
        ]
    }

    # Save temporary pipeline file
    temp_json = dem_dir / f"{las_path.stem}_pipeline.json"
    with open(temp_json, "w") as f:
        json.dump(pipeline_dict, f)

    # Run PDAL using the JSON pipeline
    cmd = ["pdal", "pipeline", str(temp_json)]
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Clean up JSON file
    temp_json.unlink(missing_ok=True)

    if result.returncode != 0:
        return f"{las_path.name} failed:\n{result.stderr.strip()}"
    else:
        return f"{las_path.name} done"


# parallel execution
max_workers = max(1, os.cpu_count() - 1)
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(run_pdal, las) for las in las_files]

    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing DEMs"):
        tqdm.write(f.result())


⏩ P5212G2_4.laz skipped
⏩ P5222G4_1.laz skipped
⏩ P5212G2_3.laz skipped
⏩ P5222G3_9.laz skipped
⏩ P5222C4_8.laz skipped
⏩ P5221H1_7.laz skipped
⏩ P5212G2_2.laz skipped
⏩ P5222G3_8.laz skipped
⏩ P5222C4_7.laz skipped
⏩ P5221H1_6.laz skipped
⏩ P4441H3_6.laz skipped
⏩ P5212G2_1.laz skipped
⏩ P5222G3_7.laz skipped
⏩ P5222C4_6.laz skipped
⏩ P5221H1_5.laz skipped
⏩ P5212G1_9.laz skipped
⏩ P5222G3_6.laz skipped
⏩ P5222C4_5.laz skipped
⏩ P5221H1_4.laz skipped
⏩ P5212G1_8.laz skipped
⏩ P5222G3_5.laz skipped
⏩ P5222C4_4.laz skipped
⏩ P5221H1_3.laz skipped
⏩ P5212G1_7.laz skipped


Processing DEMs:   3%|▎         | 41/1600 [00:00<00:14, 105.81it/s]

⏩ P4441H3_4.laz skipped
⏩ P5222G3_4.laz skipped
⏩ P5222C4_3.laz skipped
⏩ P5221H1_2.laz skipped
⏩ P5212G1_6.laz skipped
⏩ P5222G3_3.laz skipped
⏩ P5222C4_2.laz skipped
⏩ P5221H1_1.laz skipped
⏩ P5212G1_5.laz skipped
⏩ P4441H3_3.laz skipped
⏩ P5222G3_2.laz skipped
⏩ P5222C4_1.laz skipped
⏩ P5221G4_9.laz skipped
⏩ P5212G1_4.laz skipped
⏩ P5222G3_1.laz skipped
⏩ P5222C3_9.laz skipped
⏩ P5221G4_8.laz skipped


⏩ P4441G4_2.laz skipped
⏩ P5212G1_3.laz skipped
⏩ P4441H3_2.laz skipped
⏩ P5222G2_9.laz skipped
⏩ P5222C3_8.laz skipped
⏩ P5221G4_7.laz skipped
⏩ P5212G1_2.laz skipped
⏩ P4441G4_1.laz skipped
⏩ P5222G2_8.laz skipped
⏩ P5222C3_7.laz skipped
⏩ P5221G4_6.laz skipped
⏩ P5212G1_1.laz skipped
⏩ P4441G3_9.laz skipped
⏩ P5222G2_7.laz skipped


Processing DEMs:   5%|▍         | 73/1600 [00:00<00:16, 91.90it/s]

...

P5212A2_2.laz done


Processing DEMs:  95%|█████████▌| 1521/1600 [35:12<22:19, 16.95s/it]

P5212A2_3.laz done


Processing DEMs:  95%|█████████▌| 1522/1600 [35:27<21:07, 16.25s/it]

P5212A2_4.laz done


### Feature Enhancement and Label Generation (HPMF & Rasterization)

In [8]:
# Load vector data (ditch lines) from GeoPackage
# !!! IMPORTANT: Replace with the path to your own vector dataset !!!
vector_path = Path("data/vector_data/Hytky_iisalmi.gpkg")
label_vector_gdf = gpd.read_file(vector_path)

In [ ]:
def minmax_normalized_image(image):
    # Handle uniform images: if all values are equal, return a zero array to avoid division by zero
    if np.max(image) == np.min(image):
        return np.zeros(image.shape, dtype=np.float32)

    # Replace no data values with ones
    image = np.where(np.isnan(image) | (image == -9999), 1, image)

    scaler = MinMaxScaler()                                               # Initialize MinMaxScaler to scale pixel values between 0 and 1
    flat_normalized_image = scaler.fit_transform(image.reshape(-1, 1))    # Flatten the image for scaler input and apply normalization
    normalized_image = flat_normalized_image.reshape(image.shape)         # Reshape the normalized data back to the original image dimensions

    return normalized_image.astype(np.float32)

def majority_filter(arr, size=3):
    """Apply a 2D majority filter (mode of neighborhood) using SciPy."""
    def majority(window):
        mode = stats.mode(window, keepdims=True)[0]
        return mode[0]
    return generic_filter(arr, function=majority, size=size)

In [19]:
import os, sys
os.environ["PATH"] = ";".join([
    p for p in os.environ["PATH"].split(";")
    if "Python31" not in p and "Python312" not in p and "Python313" not in p
])
import cv2
print(cv2.__version__)

ImportError: DLL load failed while importing cv2: Uvedená procedura nebyla nalezena.

In [21]:
tile_size = 512
original_raster_size = 2048
label_hpmf_threshold = 0.0
minimum_ditch_pixel_percentage = 0.1  # %
buffer_distance = 1.5
tile_idx = 0
# Iterate through all DEM data
# for dem in dem_dir.iterdir():
from itertools import islice

for dem in islice(dem_dir.iterdir(), 10):
    if not dem.is_file():
        continue
    # Apply High Pass Median Filter (HPMF) to DEM
    # hpmf_file = hpmf_dir / f"{dem.stem}_hpmf.tif"
    # wbt.high_pass_median_filter(i=dem, output=hpmf_file, filterx=11, filtery=11)
    with rasterio.open(dem) as src:
        dem_array = src.read(1)
        dem_bounds = src.bounds
        dem_transform = src.transform
        dem_shape = src.shape
        meta = src.meta.copy()
    median = median_filter(dem_array, size=11)
    hpmf_array = dem_array - median
    dem_geom = box(dem_bounds.left, dem_bounds.bottom, dem_bounds.right, dem_bounds.top)
    dem_gdf = gpd.GeoDataFrame(geometry=[dem_geom], crs=label_vector_gdf.crs)

    # # Create a polygon covering the HPMF tile extent
    # hpmf_geom = box(hpmf_bounds.left, hpmf_bounds.bottom, hpmf_bounds.right, hpmf_bounds.top)
    # hpmf_gdf = gpd.GeoDataFrame(geometry=[hpmf_geom], crs=label_vector_gdf.crs)

    # Clip vector data (ditches) to HPMF tile extent
    clipped_gdf = gpd.clip(gdf=label_vector_gdf, mask=dem_gdf)
    if clipped_gdf.empty:
        print("  No vector features in this DEM — skipping.")
        continue

    # Buffer vector geometries (1.5 m) to give them width
    buffered_label_geom = clipped_gdf.buffer(distance=buffer_distance)

    # Rasterize buffered geometries onto HPMF tile grid
    buffered_label_array = features.rasterize(shapes=[(geom, 1) for geom in buffered_label_geom.geometry], # Geometries to rasterize (value=1 inside buffer)
                                              out_shape=dem_shape,                                        # Match output size to input raster
                                              transform=dem_transform,                                         # Align to same grid/coordinates as input
                                              fill=0,                                                      # Background pixels get value 0
                                              dtype=np.uint8,                                              # Use 8-bit integer values
                                              all_touched=True)                                            # Mark all pixels touched by geometry, not just centers)

    # Combine buffered vector raster with HPMF mask
    # Keep only pixels within buffer where HPMF < 0.00
    final_label_array = np.where((buffered_label_array == 1) & (hpmf_array < label_hpmf_threshold), 1, 0)

    final_label_array = majority_filter(final_label_array, size=3)

    # --- Normalize & resample both rasters ---
    hpmf_array = minmax_normalized_image(hpmf_array)
    hpmf_array = cv2.resize(hpmf_array, (original_raster_size, original_raster_size), interpolation=cv2.INTER_LINEAR)
    label_array = cv2.resize(final_label_array, (original_raster_size, original_raster_size), interpolation=cv2.INTER_NEAREST)
    # --- Generate 512×512 tiles ---
    for i in range(0, original_raster_size, tile_size):
        for j in range(0, original_raster_size, tile_size):
            label_tile = label_array[i:i + tile_size, j:j + tile_size]
            ditch_ratio = np.mean(label_tile == 1) * 100
            if ditch_ratio < minimum_ditch_pixel_percentage:
                continue  # skip nearly empty tiles

            hpmf_tile = hpmf_array[i:i + tile_size, j:j + tile_size]

            label_tile_file = label_dir / f"{tile_idx}.tif"
            hpmf_tile_file = hpmf_dir / f"{tile_idx}.tif"

            with rasterio.open(
                label_tile_file,
                'w',
                driver='GTiff',
                height=label_tile.shape[0],
                width=label_tile.shape[1],
                count=1,
                dtype=label_tile.dtype,
                crs=src.crs,  
                transform=src.transform,
            ) as dst:
                dst.write(label_tile, 1)
            
            with rasterio.open(
                hpmf_tile_file,
                'w',
                driver='GTiff',
                height=hpmf_tile.shape[0],
                width=hpmf_tile.shape[1],
                count=1,
                dtype=hpmf_tile.dtype,
                crs=src.crs,
                transform=src.transform,
            ) as dst:
                dst.write(hpmf_tile, 1)
            tile_idx += 1

print(f"✅ Finished — {tile_idx} tiles generated.")


ImportError: DLL load failed while importing cv2: Uvedená procedura nebyla nalezena.

### Global Robust Normalization

To efficiently compute global robust statistics, a subset of valid pixels is sampled from each raster tile instead of loading all data into memory. Each tile is read block by block, and invalid values (e.g., NaN, -9999) are masked out. Up to a fixed number (e.g., 5000) of valid pixels are randomly selected from each file. All sampled pixels are aggregated into a global pool, from which percentiles (p1, p99) and the median are computed. 


In [8]:
# ----------------------------------------------------------------------
# Parameters
# ----------------------------------------------------------------------
MAX_SAMPLES_PER_TILE = 5000     
P_LO, P_HI = 1, 99              
EPS = 1e-6                      
OUTPUT_NODATA = -9999.0         
NUM_WORKERS = max(1, os.cpu_count() - 1)
rng = np.random.default_rng(2025)

# ----------------------------------------------------------------------
# Utility functions
# ----------------------------------------------------------------------
def read_blocks_randomized(src):
    blocks = list(src.block_windows(1))
    rng.shuffle(blocks)
    for ji, window in blocks:             
        arr = src.read(1, window=window)   
        yield window, arr

def mask_nodata(arr, nodata_val):
    mask = np.ones(arr.shape, dtype=bool)
    if nodata_val is not None and not (isinstance(nodata_val, float) and math.isnan(nodata_val)):
        mask &= (arr != nodata_val)
    mask &= (arr > -1e3)  
    mask &= ~np.isnan(arr)
    return mask

def sample_tile_values(tif_path, per_tile_cap=MAX_SAMPLES_PER_TILE):
    samples, remaining = [], per_tile_cap
    with rasterio.open(tif_path) as src:
        nodata_val = src.nodata
        for _, block in read_blocks_randomized(src):
            block = block.astype("float32", copy=False)
            valid = mask_nodata(block, nodata_val)
            if not valid.any(): 
                continue
            vals = block[valid]
            if vals.size <= remaining:
                samples.append(vals); remaining -= vals.size
            else:
                idx = rng.choice(vals.size, size=remaining, replace=False)
                samples.append(vals[idx]); remaining = 0
            if remaining <= 0:
                break
    return np.concatenate(samples) if samples else np.empty((0,), dtype="float32")

def compute_global_params(tif_list):
    all_samples = []
    for i, p in enumerate(tif_list, 1):
        s = sample_tile_values(p)
        if s.size:
            all_samples.append(s)
        if i % 10 == 0 or i == len(tif_list):
            print(f"[Pass1] Sampled {i}/{len(tif_list)} files")
    if not all_samples:
        raise RuntimeError("No valid samples collected. Check NoData handling or input path.")
    pooled = np.concatenate(all_samples)
    g_p1 = float(np.percentile(pooled, P_LO))
    g_p99 = float(np.percentile(pooled, P_HI))
    g_med = float(np.median(pooled))
    print(f"\n[Global] p{P_LO}={g_p1:.6f}, median={g_med:.6f}, p{P_HI}={g_p99:.6f} (sample size={pooled.size:,})")
    return g_p1, g_p99, g_med

def normalize_block(block, p1, p99, med):
    out = block.astype("float32", copy=True)
    np.clip(out, p1, p99, out=out)
    scale = max(p99 - p1, EPS)
    out = (out - med) / scale
    return out

def normalize_one_file(src_path, dst_path, p1, p99, med):
    with rasterio.open(src_path) as src:
        profile = src.profile
        nodata_val = src.nodata
        profile.update(
            dtype="float32",
            count=1,
            nodata=OUTPUT_NODATA,
            compress="deflate", 
            predictor=3,         
            tiled=False,
            bigtiff="IF_SAFER"
        )
        with rasterio.open(dst_path, "w", **profile) as dst:
          for ji, window in src.block_windows(1):                 
                arr = src.read(1, window=window).astype("float32", copy=False)
                valid = mask_nodata(arr, nodata_val)
                tile = np.full(arr.shape, OUTPUT_NODATA, dtype="float32")
                if valid.any():
                    norm = normalize_block(arr, p1, p99, med)
                    tile[valid] = norm[valid]
                dst.write(tile, 1, window=window)
 


In [9]:
# ----------------------------------------------------------------------
# Main routine
# ----------------------------------------------------------------------
def run_global_robust_normalization(hpmf_dir: Path, out_dir: Path):
    tif_list = sorted(hpmf_dir.glob("*.tif"))
    if not tif_list:
        raise FileNotFoundError(f"No GeoTIFF found in: {hpmf_dir}")

    print(f"Found {len(tif_list)} files. Starting Pass 1 (sampling)...")
    p1, p99, med = compute_global_params(tif_list)

    params_path = out_dir / "global_norm_params.json"
    with params_path.open("w") as f:
        json.dump({"p1": p1, "median": med, "p99": p99, "P_LO": P_LO, "P_HI": P_HI}, f, indent=2)
    print(f"Saved global parameters to: {params_path}")

    print(f"\nStarting Pass 2 (normalize & write) -> {out_dir}")
    tasks = []
    for p in tif_list:
        dst = out_dir / f"{p.stem}_normalized.tif"
        tasks.append((p, dst))

    done = 0
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = {ex.submit(normalize_one_file, str(src), str(dst), p1, p99, med): (src, dst) for src, dst in tasks}
        for fut in as_completed(futures):
            _ = fut.result() 
            done += 1
            if done % 10 == 0 or done == len(tasks):
                print(f"[Pass2] Normalized {done}/{len(tasks)}")

    index_csv = normalized_dir / "normalized_index.csv"
    with index_csv.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["source", "normalized"])
        for _, dst in tasks:
            w.writerow([str(_), str(dst)])
    print(f"\nAll normalization done.")
    print(f"Index CSV: {index_csv}")
    print(f"Output dir: {normalized_dir}")

# Run
run_global_robust_normalization(hpmf_dir, normalized_dir)

Found 3 files. Starting Pass 1 (sampling)...
[Pass1] Sampled 3/3 files

[Global] p1=-0.230000, median=0.000000, p99=0.190000 (sample size=15,000)
Saved global parameters to: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles\global_norm_params.json

Starting Pass 2 (normalize & write) -> C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles
[Pass2] Normalized 3/3

All normalization done.
Index CSV: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles\normalized_index.csv
Output dir: C:\Users\Matěj\Documents\GitHub\GIS_E6010_Project_Course_2025\preprocessing\model_input_data\normalized_tiles
